# Inter-Annotator-Agreement: Wortartenebene

**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus
`03_annotation_laufend/pos/tranche<N>/` sowie aus `02_quelldaten/`,
schreibt nach `05_ergebnisse/iaa/`.

## Konfiguration


In [ ]:
from pathlib import Path

TRANCHE = 1
BASIS_ARBEIT = (Path("..") / "03_annotation_laufend" / "pos"
                / f"tranche{TRANCHE}")
BASIS_QUELL  = Path("..") / "02_quelldaten"
CONLL        = BASIS_QUELL / "goldstandard_v1.conll"
AUSGABE      = Path("..") / "05_ergebnisse" / "iaa"

LAUF_ETIKETT = "vor Adjudikation"

RICHTLINIE_VERSION = "1.18"
BOOTSTRAP_REPS, SEED = 2000, 42

GROB = {"ADJA": "ADJ", "ADJD": "ADJ"}

AUSGABE.mkdir(parents=True, exist_ok=True)
assert CONLL.exists(), f"{CONLL} nicht gefunden."

gefunden = []
for endung in ("xlsx", "ods"):
    gefunden += [q for q in BASIS_ARBEIT.glob(f"*_pos_tranche*.{endung}")
                 if not q.name.startswith(("~$", "."))]
mappen = {}
for q in sorted(gefunden, key=lambda x: x.name):
    person = q.name.split("_")[0]
    if person in mappen:
        raise ValueError(
            f"Fuer {person} liegen zwei Dateien vor: "
            f"{mappen[person].name} und {q.name}. Bitte eine entfernen.")
    mappen[person] = q
assert len(mappen) >= 2, f"Mindestens zwei Mappen noetig, gefunden: {list(mappen)}"

LAUF_NR = len(sorted(AUSGABE.glob(f"iaa_pos_t{TRANCHE}_lauf*.json"))) + 1
print(f"Tranche {TRANCHE} | Bearbeitende ({len(mappen)}):")
for person, q in mappen.items():
    print(f"  {person:<5} {q.name}")
print(f"Dieser Durchlauf: Lauf {LAUF_NR} ({LAUF_ETIKETT})")

Tranche 1 | Bearbeitende (3):
  A4    A4_pos_tranche1_final.xlsx
  A5    A5_pos_tranche1_final.xlsx
  A6    A6_pos_tranche1_final.ods
Dieser Durchlauf: Lauf 1 (vor Adjudikation)


## Lesepakete prüfen


In [ ]:
import importlib.util

benoetigt = {"xlsx": ("openpyxl", "openpyxl"), "ods": ("odf", "odfpy")}
fehlt = []
for endung, (modul, paket) in benoetigt.items():
    vorhanden = any(q.suffix == f".{endung}" for q in mappen.values())
    if vorhanden and importlib.util.find_spec(modul) is None:
        fehlt.append((endung, paket))

if fehlt:
    print("FEHLENDE PAKETE\n")
    for endung, paket in fehlt:
        print(f"  Fuer die vorliegenden .{endung}-Dateien wird '{paket}' benoetigt.")
    print("\nInstallation:")
    print("  pip install " + " ".join(paket for _, paket in fehlt))
    print("\nDanach den Kernel neu starten und dieses Notebook erneut ausfuehren.")
    raise SystemExit("Bitte die genannten Pakete installieren.")
print("Lesepakete vorhanden.")


Lesepakete vorhanden.


## Schritt 1 — Maße implementieren und selbst validieren

In [ ]:
from collections import Counter
from itertools import permutations, combinations

def prozent_agreement(einheiten):
    werte = []
    for urteile in einheiten:
        u = [x for x in urteile if x is not None]
        paare = list(permutations(range(len(u)), 2))
        werte.append(sum(1 for i, j in paare if u[i] == u[j]) / len(paare))
    return sum(werte) / len(werte)


def einheit_agreement(urteile):
    u = [x for x in urteile if x is not None]
    paare = list(permutations(range(len(u)), 2))
    return sum(1 for i, j in paare if u[i] == u[j]) / len(paare)


def fleiss_kappa(einheiten):
    umfaenge = {len([x for x in u if x is not None]) for u in einheiten}
    if len(umfaenge) > 1:
        raise ValueError(
            "Fleiss' kappa verlangt gleich viele Urteile je Einheit; "
            f"vorgefunden: {sorted(umfaenge)}. Mappen pruefen.")
    p_o = prozent_agreement(einheiten)
    zaehler, gesamt = Counter(), 0
    for urteile in einheiten:
        for x in (v for v in urteile if v is not None):
            zaehler[x] += 1
            gesamt += 1
    p_e = sum((v / gesamt) ** 2 for v in zaehler.values())
    return ((p_o - p_e) / (1 - p_e) if p_e < 1 else float("nan")), p_o, p_e


def kategorienverteilung(einheiten):
    zaehler, gesamt = Counter(), 0
    for urteile in einheiten:
        for x in (v for v in urteile if v is not None):
            zaehler[x] += 1
            gesamt += 1
    return {k: v / gesamt for k, v in zaehler.most_common()}


def lesehilfe_kappa(einheiten, kappa, beobachtet, erwartet, einrueckung=""):
    vert = kategorienverteilung(einheiten)
    gezeigt = list(vert.items())[:6]
    rest = len(vert) - len(gezeigt)
    print(f"{einrueckung}Kategorien ({len(vert)}): "
          + " | ".join(f"{k} {100 * v:.1f}%" for k, v in gezeigt)
          + (f" | ... und {rest} weitere" if rest > 0 else ""))
    print(f"{einrueckung}erwartet = Summe der quadrierten Anteile = "
          f"{erwartet:.4f}")
    print(f"{einrueckung}kappa = ({beobachtet:.4f} - {erwartet:.4f}) / "
          f"(1 - {erwartet:.4f}) = {beobachtet - erwartet:.4f} / "
          f"{1 - erwartet:.4f} = {kappa:.4f}")


def fleiss_gegenprobe(einheiten):
    try:
        import numpy as np
        from statsmodels.stats.inter_rater import (aggregate_raters,
                                                   fleiss_kappa as sm_fleiss)
    except ImportError:
        return None
    tabelle, _ = aggregate_raters(np.array(einheiten))
    #method ausdruecklich setzen: die Alternative "randolph" unterstellt
    #eine Gleichverteilung der Kategorien und ergibt ein anderes Mass.
    return float(sm_fleiss(tabelle, method="fleiss"))


# Anker: Fleiss, 30 Einheiten und 6 Bearbeitende, wie im
# Testfall der Bibliothek DKPro Agreement
referenz = [(4, 4, 4, 4, 4, 4), (2, 2, 2, 5, 5, 5), (2, 3, 3, 3, 3, 5),
            (5, 5, 5, 5, 5, 5), (2, 2, 2, 4, 4, 4), (1, 1, 3, 3, 3, 3),
            (3, 3, 3, 3, 5, 5), (1, 1, 3, 3, 3, 4), (1, 1, 4, 4, 4, 4),
            (5, 5, 5, 5, 5, 5), (1, 4, 4, 4, 4, 4), (1, 2, 4, 4, 4, 4),
            (2, 2, 2, 3, 3, 3), (1, 4, 4, 4, 4, 4), (2, 2, 4, 4, 4, 5),
            (3, 3, 3, 3, 3, 5), (1, 1, 1, 4, 5, 5), (1, 1, 1, 1, 1, 2),
            (2, 2, 4, 4, 4, 4), (1, 3, 3, 5, 5, 5), (5, 5, 5, 5, 5, 5),
            (2, 4, 4, 4, 4, 4), (2, 2, 4, 5, 5, 5), (1, 1, 4, 4, 4, 4),
            (1, 4, 4, 4, 4, 5), (2, 2, 2, 2, 2, 4), (1, 1, 1, 1, 5, 5),
            (2, 2, 4, 4, 4, 4), (1, 3, 3, 3, 3, 3), (5, 5, 5, 5, 5, 5)]
fk, po, pe = fleiss_kappa(referenz)
print("Selbstvalidierung (Soll: 0,5556 / 0,2201 / 0,430)")
print(f"  beobachtet {po:.4f} | erwartet {pe:.4f} | Fleiss kappa {fk:.4f}")
for wert, soll, name in ((po, 0.5556, "beobachtet"),
                         (pe, 0.2201, "erwartet"), (fk, 0.430, "kappa")):
    assert abs(wert - soll) < 0.001, f"{name}: {wert:.4f} statt {soll}"

fremd = fleiss_gegenprobe(referenz)
if fremd is None:
    print("  Gegenprobe uebersprungen: statsmodels nicht installiert")
else:
    assert abs(fk - fremd) < 1e-9, (fk, fremd)
    print(f"  Gegenprobe statsmodels: {fremd:.4f} — uebereinstimmend")

try:
    fleiss_kappa([(1, 1, 1), (1, 1)])
except ValueError:
    print("  Ungleiche Urteilszahlen werden erkannt")
else:
    raise AssertionError("ungleiche Urteilszahlen blieben unbemerkt")
# Anker 2 und 3: Grenzfaelle, deren Ergebnis feststeht
#   identische Reihen bei zwei Kategorien -> kappa = 1
#   Uebereinstimmung genau auf Zufallsniveau -> kappa = 0
assert abs(fleiss_kappa([("a", "a"), ("b", "b"),
                         ("a", "a"), ("b", "b")])[0] - 1.0) < 1e-9
assert abs(fleiss_kappa([("x", "x"), ("x", "y"),
                         ("y", "x"), ("y", "y")])[0]) < 1e-9
print("  Fleiss kappa: Grenzfaelle 1,0 und 0,0 reproduziert")

# Anker 4: Uebereinstimmung innerhalb einer einzelnen Einheit. Diese
# Funktion bestimmt spaeter, welche Einheiten als einstimmig gelten und
# damit, was in die Divergenz- und Kommentarlisten gelangt.
assert einheit_agreement(("a", "a", "a")) == 1.0
assert abs(einheit_agreement(("a", "a", "b")) - 1 / 3) < 1e-9
assert einheit_agreement(("a", "b", "c")) == 0.0
print("  einheit_agreement: 1,000 / 0,333 / 0,000 korrekt")

# Anker: Kategorienverteilung, aus der die erwartete Uebereinstimmung folgt
_v = kategorienverteilung([("a", "a"), ("a", "b")])
assert abs(_v["a"] - 0.75) < 1e-9 and abs(_v["b"] - 0.25) < 1e-9
assert abs(sum(x * x for x in _v.values()) - 0.625) < 1e-9
print("  kategorienverteilung: 0,75 / 0,25 korrekt")
print()
print("Alle Anker getroffen — die Implementierung darf verwendet werden.")

Selbstvalidierung (Soll: 0,5556 / 0,2201 / 0,430)
  beobachtet 0.5556 | erwartet 0.2199 | Fleiss kappa 0.4302
  Gegenprobe statsmodels: 0.4302 — uebereinstimmend
  Ungleiche Urteilszahlen werden erkannt
  Fleiss kappa: Grenzfaelle 1,0 und 0,0 reproduziert
  einheit_agreement: 1,000 / 0,333 / 0,000 korrekt
  kategorienverteilung: 0,75 / 0,25 korrekt

Alle Anker getroffen — die Implementierung darf verwendet werden.


## Schritt 2 — Mappen laden und über die Token-Kennung ausrichten


In [4]:
import pandas as pd

tabellen = {}
for person, pfad in mappen.items():
    df = pd.read_excel(pfad, sheet_name="POS", dtype=str).fillna("")
    df["id"] = df["id"].astype(int)
    tabellen[person] = df.sort_values("id").reset_index(drop=True)
    print(f"  {person}: {len(df)} Zeilen")

personen = list(tabellen)
basis = tabellen[personen[0]]
for person, df in tabellen.items():
    assert list(df["id"]) == list(basis["id"]), \
        f"{person} hat eine andere Tokenmenge."
    assert list(df["token"]) == list(basis["token"]), \
        f"{person} fuehrt bei gleicher id ein anderes Token."
print(f"\nAusrichtung ueber 'id' geprueft: {len(basis)} Token, "
      f"in allen Mappen deckungsgleich.")

#Erzaehlung und Sammlung je Token-Kennung aus dem Goldstandard holen
herkunft, lauf_id, meta = {}, 0, {}
for z in CONLL.read_text(encoding="utf-8").splitlines():
    if z.startswith("#"):
        if " = " in z:
            k, w = z[1:].split(" = ", 1)
            meta[k.strip()] = w.strip()
        continue
    if not z.strip():
        continue
    herkunft[lauf_id] = (meta.get("erzaehlung", "?"), meta.get("sammlung", "?"))
    lauf_id += 1

daten = pd.DataFrame({"id": basis["id"], "token": basis["token"]})
daten["erzaehlung"] = [herkunft[i][0] for i in daten["id"]]
daten["sammlung"]   = [herkunft[i][1] for i in daten["id"]]
for person in personen:
    daten[person] = tabellen[person]["tag"].str.strip().str.upper().values
    daten["unsicher_" + person] = (
        tabellen[person]["unsicher"].str.strip().str.lower() != "")

luecken = {p: int((daten[p] == "").sum()) for p in personen}
print("Noch nicht getaggt: " + ", ".join(f"{p}={n}" for p, n in luecken.items()))
vollstaendig = daten[(daten[personen] != "").all(axis=1)].copy()
print(f"Messgrundlage (von allen getaggt): {len(vollstaendig)} von {len(daten)}")
assert len(vollstaendig) > 0, "Keine gemeinsam bearbeiteten Token."

  A4: 1794 Zeilen
  A5: 1794 Zeilen
  A6: 1794 Zeilen

Ausrichtung ueber 'id' geprueft: 1794 Token, in allen Mappen deckungsgleich.
Noch nicht getaggt: A4=0, A5=0, A6=0
Messgrundlage (von allen getaggt): 1794 von 1794


## Schritt 3 — Übereinstimmung auf beiden Granularitäten


In [5]:
def zu_grob(tag):
    return GROB.get(tag, tag)

for person in personen:
    vollstaendig["grob_" + person] = vollstaendig[person].map(zu_grob)

sp_fein = personen
sp_grob = ["grob_" + p for p in personen]

def kennzahlen(spalten):
    einheiten = [tuple(r) for r in vollstaendig[spalten].to_numpy()]
    fk_w, po_w, pe_w = fleiss_kappa(einheiten)
    einstimmig = sum(1 for u in einheiten if len(set(u)) == 1)
    return dict(n=len(einheiten), prozent=po_w, fleiss_kappa=fk_w,
                beobachtet=po_w, erwartet=pe_w, einstimmig=einstimmig,
                uneinig=len(einheiten) - einstimmig)

grob = kennzahlen(sp_grob)
fein = kennzahlen(sp_fein)

print(f"{'Ebene':<34}{'n':>7}{'PA':>9}{'Fleiss K':>10}{'uneinig':>9}")
print(f"{'zusammengefasst (Hauptwert)':<34}{grob['n']:>7}"
      f"{grob['prozent']:>9.3f}{grob['fleiss_kappa']:>10.3f}"
      f"{grob['uneinig']:>9}")
print(f"{'fein (mit ADJA/ADJD)':<34}{fein['n']:>7}"
      f"{fein['prozent']:>9.3f}{fein['fleiss_kappa']:>10.3f}"
      f"{fein['uneinig']:>9}")
print(f"{'erwartete Uebereinstimmung':<34}{'':>7}"
      f"{grob['erwartet']:>9.3f}{'':>10}{'':>9}")

einheiten_grob = [tuple(r) for r in vollstaendig[sp_grob].to_numpy()]
print()
lesehilfe_kappa(einheiten_grob, grob["fleiss_kappa"], grob["beobachtet"],
                grob["erwartet"], einrueckung="  ")

gegen = fleiss_gegenprobe(einheiten_grob)
if gegen is None:
    print("\nGegenprobe uebersprungen: statsmodels nicht installiert")
else:
    assert abs(grob["fleiss_kappa"] - gegen) < 1e-9
    print(f"\nGegenprobe statsmodels (Hauptwert): {gegen:.3f} "
          f"— uebereinstimmend")
print()
print(f"Zusaetzliche Uneinigkeit allein durch die Adjektiv-"
      f"Feindifferenzierung: {fein['uneinig'] - grob['uneinig']} Token")
print()
print("Der Hauptwert ist die zusammengefasste Ebene; sie entspricht der")
print("Granularitaet, auf der das Tagger-Ensemble bewertet wird.")

Ebene                                   n       PA  Fleiss K  uneinig
zusammengefasst (Hauptwert)          1794    0.907     0.896      240
fein (mit ADJA/ADJD)                 1794    0.905     0.894      245
erwartete Uebereinstimmung                   0.103                   

  Kategorien (15): NOUN 15.0% | PUNCT 14.5% | VERB 13.0% | DET 11.7% | PRON 10.8% | ADP 6.5% | ... und 9 weitere
  erwartet = Summe der quadrierten Anteile = 0.1031
  kappa = (0.9071 - 0.1031) / (1 - 0.1031) = 0.8040 / 0.8969 = 0.8964

Gegenprobe statsmodels (Hauptwert): 0.896 — uebereinstimmend

Zusaetzliche Uneinigkeit allein durch die Adjektiv-Feindifferenzierung: 5 Token

Der Hauptwert ist die zusammengefasste Ebene; sie entspricht der
Granularitaet, auf der das Tagger-Ensemble bewertet wird.


## Schritt 4 — Adjektiv-Feindifferenzierung für sich betrachtet


In [ ]:
ist_adj = vollstaendig[sp_fein].isin(["ADJA", "ADJD"]).any(axis=1)
adj = vollstaendig[ist_adj].copy()
print(f"Token, die mindestens eine Person als Adjektiv einstuft: {len(adj)}")

if len(adj) == 0:
    adj_werte = {}
    print("Keine Adjektive in der Messgrundlage — Auswertung entfaellt.")
else:
    #a) Einigkeit darueber, DASS ein Adjektiv vorliegt
    grob_einig = sum(1 for r in adj[["grob_" + p for p in personen]].to_numpy()
                     if len(set(r)) == 1)
    #b) unter den darueber Einigen: Einigkeit ueber ADJA gegen ADJD
    einig_adj = adj[[len(set(r)) == 1 for r in
                     adj[["grob_" + p for p in personen]].to_numpy()]]
    einig_adj = einig_adj[einig_adj["grob_" + personen[0]] == "ADJ"]
    if len(einig_adj):
        einheiten = [tuple(r) for r in einig_adj[sp_fein].to_numpy()]
        fk_adj, pa_adj, _ = fleiss_kappa(einheiten)
        uneinig_adj = sum(1 for u in einheiten if len(set(u)) > 1)
        n_adj = len(einig_adj)
        z_ = 1.96
        nen = 1 + z_ * z_ / n_adj
        zen = (pa_adj + z_ * z_ / (2 * n_adj)) / nen
        rnd = z_ * ((pa_adj * (1 - pa_adj) / n_adj
                     + z_ * z_ / (4 * n_adj * n_adj)) ** 0.5) / nen
        wilson_adj = [zen - rnd, zen + rnd]
    else:
        pa_adj = fk_adj = float("nan")
        uneinig_adj, wilson_adj = 0, [float("nan")] * 2

    adj_werte = dict(n_kandidaten=int(len(adj)),
                     n_einig_dass_adjektiv=int(len(einig_adj)),
                     prozent=float(pa_adj), fleiss_kappa=float(fk_adj),
                     wilson_95=[float(w) for w in wilson_adj],
                     uneinig=int(uneinig_adj))

    print(f"  davon alle einig, DASS ein Adjektiv vorliegt: {len(einig_adj)}"
          f"   <- Fallzahl dieser Kennzahl")
    print(f"  Uebereinstimmung ADJA gegen ADJD in dieser Gruppe:")
    print(f"    n = {len(einig_adj)} | PA {pa_adj:.3f} "
          f"| Fleiss kappa {fk_adj:.3f} | uneinig {uneinig_adj}")
    print(f"    Wilson-95 fuer PA: [{wilson_adj[0]:.3f}, {wilson_adj[1]:.3f}]")
    print(f"    Die Fallzahl liegt deutlich unter der des Hauptwerts "
          f"({len(vollstaendig)});")
    print(f"    ohne das Intervall wuerde die Kennzahl ueberinterpretiert.")
    print()
    print("Verteilung der Adjektivtags je Person:")
    for p in personen:
        c = Counter(adj[p])
        print(f"  {p}: ADJA={c.get('ADJA', 0)}, ADJD={c.get('ADJD', 0)}, "
              f"anderes={sum(v for k, v in c.items() if k not in ('ADJA', 'ADJD'))}")

    if len(einig_adj) >= 10:
        print()
        print("Paarweise Uebereinstimmung, beschraenkt auf Adjektive:")
        adj_paare, adj_index = {}, {}
        for a, b in combinations(personen, 2):
            pa_ab = (einig_adj[a] == einig_adj[b]).mean()
            adj_paare[f"{a}-{b}"] = float(pa_ab)
            print(f"  {a}-{b:<8} {pa_ab:.3f}")
        for p in personen:
            werte = [v for k, v in adj_paare.items() if p in k.split("-")]
            adj_index[p] = sum(werte) / len(werte)
        spanne_adj = max(adj_index.values()) - min(adj_index.values())
        print("  Abweichungsindex auf Adjektiven:")
        for p, v in sorted(adj_index.items(), key=lambda x: x[1]):
            print(f"    {p:<5} {v:.3f}")
        if spanne_adj > 0.05:
            print(f"  Hinweis: {min(adj_index, key=adj_index.get)} trifft die "
                  f"Unterscheidung abweichend (Spanne {spanne_adj:.3f}).")
            print("  Diese Schwaeche bleibt im Gesamtwert unsichtbar, weil "
                  "Adjektive nur")
            print("  einen kleinen Teil der Token stellen.")
        else:
            print(f"  Kein Ausreisser auf Adjektivebene (Spanne "
                  f"{spanne_adj:.3f}).")
        adj_werte["paarweise"] = adj_paare
        adj_werte["abweichungsindex"] = {k: float(v)
                                         for k, v in adj_index.items()}

Token, die mindestens eine Person als Adjektiv einstuft: 122
  davon alle einig, DASS ein Adjektiv vorliegt: 90   <- Fallzahl dieser Kennzahl
  Uebereinstimmung ADJA gegen ADJD in dieser Gruppe:
    n = 90 | PA 0.963 | Fleiss kappa 0.902 | uneinig 5
    Wilson-95 fuer PA: [0.901, 0.987]
    Die Fallzahl liegt deutlich unter der des Hauptwerts (1794);
    ohne das Intervall wuerde die Kennzahl ueberinterpretiert.

Verteilung der Adjektivtags je Person:
  A4: ADJA=65, ADJD=31, anderes=26
  A5: ADJA=78, ADJD=40, anderes=4
  A6: ADJA=78, ADJD=24, anderes=20

Paarweise Uebereinstimmung, beschraenkt auf Adjektive:
  A4-A5       0.978
  A4-A6       0.944
  A5-A6       0.967
  Abweichungsindex auf Adjektiven:
    A6    0.956
    A4    0.961
    A5    0.972
  Kein Ausreisser auf Adjektivebene (Spanne 0.017).


## Schritt 5 — Paarweise Werte und Abweichungsindex


In [ ]:
paarwerte = {}

print(f"{'Paar':<14}{'PA':>9}{'F-K = pi':>10}{'uneinig':>9}")
for a, b in combinations(personen, 2):
    va = list(vollstaendig["grob_" + a])
    vb = list(vollstaendig["grob_" + b])
    pa_ab = sum(1 for x, y in zip(va, vb) if x == y) / len(va)
    fk_ab, _, _ = fleiss_kappa(list(zip(va, vb)))
    un = sum(1 for x, y in zip(va, vb) if x != y)
    paarwerte[f"{a}-{b}"] = dict(prozent=pa_ab, fleiss_kappa=fk_ab,
                                 uneinig=un)
    print(f"{a + '-' + b:<14}{pa_ab:>9.3f}{fk_ab:>10.3f}{un:>9}")

index = {}
if len(personen) > 2:
    print()
    print("Abweichungsindex (mittlere Uebereinstimmung mit den anderen):")
    for p in personen:
        werte = [v["prozent"] for k, v in paarwerte.items()
                 if p in k.split("-")]
        index[p] = sum(werte) / len(werte)
    for p, v in sorted(index.items(), key=lambda x: x[1]):
        print(f"  {p:<5} {v:.3f}")
    spanne = max(index.values()) - min(index.values())
    if spanne > 0.05:
        print(f"  Hinweis: {min(index, key=index.get)} weicht deutlich ab "
              f"(Spanne {spanne:.3f}) — Richtlinienverstaendnis pruefen.")
    else:
        print(f"  Kein Ausreisser erkennbar (Spanne {spanne:.3f}).")

Paar                 PA  F-K = pi  uneinig
A4-A5             0.940     0.933      108
A4-A6             0.897     0.885      185
A5-A6             0.885     0.871      207

Abweichungsindex (mittlere Uebereinstimmung mit den anderen):
  A6    0.891
  A5    0.912
  A4    0.918
  Kein Ausreisser erkennbar (Spanne 0.028).


## Schritt 6 — Tagspezifische Werte, Verwechslungen und Unsicherheit


In [8]:
vollstaendig["einheit_pa"] = [einheit_agreement(tuple(r))
                              for r in vollstaendig[sp_grob].to_numpy()]
vollstaendig["einstimmig"] = vollstaendig["einheit_pa"] == 1.0

print("Uebereinstimmung je Tag (Zuordnung nach der ersten Bearbeitung):")
referenz_person = personen[0]
for tag, teil in vollstaendig.groupby(referenz_person):
    if len(teil) < 5:
        continue
    print(f"  {tag:<8} {teil['einheit_pa'].mean():.3f}  (n={len(teil)}, "
          f"nicht einstimmig={int((~teil['einstimmig']).sum())})")

print()
print("Haeufigste Verwechslungen (feine Ebene, paarweise gezaehlt):")
verwechslung = Counter()
for _, r in vollstaendig[~vollstaendig["einstimmig"]].iterrows():
    for a, b in combinations(personen, 2):
        if r[a] != r[b]:
            verwechslung[tuple(sorted((r[a], r[b])))] += 1
for (x, y), k in verwechslung.most_common(10):
    print(f"  {x:<8} gegen {y:<8} {k:>4}x")

print()
sp_uns = ["unsicher_" + p for p in personen]
vollstaendig["markiert"] = vollstaendig[sp_uns].any(axis=1)
markiert = vollstaendig[vollstaendig["markiert"]]
klar = vollstaendig[~vollstaendig["markiert"]]
print("Wirkung der unsicher-Kennzeichnung:")
print(f"  markiert:      n={len(markiert):>5} | "
      f"Uebereinstimmung {markiert['einheit_pa'].mean():.3f}"
      if len(markiert) else "  markiert:      keine")
print(f"  nicht markiert: n={len(klar):>5} | "
      f"Uebereinstimmung {klar['einheit_pa'].mean():.3f}")
if len(markiert):
    print("  Liegt der erste Wert deutlich niedriger, weist die "
          "Kennzeichnung zuverlaessig")
    print("  auf die strittigen Stellen hin und erfuellt ihren Zweck.")

Uebereinstimmung je Tag (Zuordnung nach der ersten Bearbeitung):
  ADJA     1.000  (n=65, nicht einstimmig=0)
  ADJD     0.871  (n=31, nicht einstimmig=6)
  ADP      0.966  (n=116, nicht einstimmig=6)
  ADV      0.574  (n=122, nicht einstimmig=72)
  AUX      0.698  (n=64, nicht einstimmig=29)
  CCONJ    0.960  (n=101, nicht einstimmig=6)
  DET      0.858  (n=226, nicht einstimmig=47)
  INTJ     0.810  (n=7, nicht einstimmig=2)
  NOUN     0.977  (n=264, nicht einstimmig=9)
  NUM      0.633  (n=10, nicht einstimmig=4)
  PART     0.964  (n=56, nicht einstimmig=3)
  PRON     0.929  (n=187, nicht einstimmig=19)
  PROPN    0.733  (n=20, nicht einstimmig=8)
  PUNCT    0.992  (n=262, nicht einstimmig=3)
  SCONJ    0.917  (n=32, nicht einstimmig=4)
  VERB     0.935  (n=231, nicht einstimmig=22)

Haeufigste Verwechslungen (feine Ebene, paarweise gezaehlt):
  AUX      gegen VERB       92x
  DET      gegen PRON       77x
  ADV      gegen PART       63x
  ADV      gegen SCONJ      32x
  ADJD     ge

## Schritt 7 — Unsicherheit der Schätzung


In [9]:
import math
import numpy as np

rng = np.random.default_rng(SEED)
gruppen = [g["einheit_pa"].to_numpy()
           for _, g in vollstaendig.groupby("erzaehlung")]
idx = rng.integers(0, len(gruppen), size=(BOOTSTRAP_REPS, len(gruppen)))
werte = np.array([np.concatenate([gruppen[i] for i in reihe]).mean()
                  for reihe in idx])
lo, hi = np.percentile(werte, [2.5, 97.5])

p_quote = vollstaendig["einheit_pa"].mean()
n_tok = len(vollstaendig)
z = 1.96
nenner = 1 + z * z / n_tok
zentrum = (p_quote + z * z / (2 * n_tok)) / nenner
rand = z * math.sqrt(p_quote * (1 - p_quote) / n_tok
                     + z * z / (4 * n_tok * n_tok)) / nenner

print(f"Uebereinstimmungsquote (Hauptebene): {p_quote:.3f}  (n = {n_tok})")
print(f"  Cluster-Bootstrap ueber {len(gruppen)} Erzaehlungen: "
      f"[{lo:.3f}, {hi:.3f}]")
print(f"  Wilson auf Tokenebene: [{zentrum - rand:.3f}, "
      f"{zentrum + rand:.3f}]  (ignoriert Klumpung)")

print()
print("Nach Sammlung:")
for s, teil in vollstaendig.groupby("sammlung"):
    print(f"  {s:<12} {teil['einheit_pa'].mean():.3f}  (n={len(teil)})")

Uebereinstimmungsquote (Hauptebene): 0.907  (n = 1794)
  Cluster-Bootstrap ueber 20 Erzaehlungen: [0.893, 0.920]
  Wilson auf Tokenebene: [0.893, 0.920]  (ignoriert Klumpung)

Nach Sammlung:
  Andersen     0.901  (n=371)
  Arabisch     0.894  (n=443)
  Grimm        0.920  (n=403)
  Japanisch    0.912  (n=577)


## Schritt 8 — Divergenzen nach Strittigkeit auflisten


In [ ]:
unstimmig = vollstaendig[~vollstaendig["einstimmig"]].copy()

def lage_und_mehrheit(zeile):
    """Wie gespalten war das Votum fuer dieses Token?"""
    stimmen = Counter(zeile[p] for p in personen)
    top, anzahl = stimmen.most_common(1)[0]
    if anzahl * 2 > len(personen):
        return "Mehrheit", top, anzahl
    if anzahl * 2 == len(personen):
        return "Patt", "", anzahl
    return "Alle uneins", "", anzahl

if len(unstimmig):
    ergebnisse = [lage_und_mehrheit(r) for _, r in unstimmig.iterrows()]
    unstimmig["lage"] = [e[0] for e in ergebnisse]
    unstimmig["mehrheit"] = [e[1] for e in ergebnisse]
    unstimmig["stimmen"] = [e[2] for e in ergebnisse]
else:
    unstimmig["lage"] = []; unstimmig["mehrheit"] = []; unstimmig["stimmen"] = []

rang = {"Alle uneins": 0, "Patt": 1, "Mehrheit": 2}
unstimmig = unstimmig.assign(_r=unstimmig["lage"].map(rang)).sort_values(
    ["_r", "id"]).drop(columns="_r")

print(f"Divergenzen: {len(unstimmig)} von {len(vollstaendig)} Token\n")
verteilung = Counter(unstimmig["lage"])
erklaerung = {
    "Alle uneins": "jede Person ein anderes Tag, keine Mehrheit",
    "Patt": "Stimmen gleich verteilt, keine Mehrheit",
    "Mehrheit": "eine Mehrheit liegt vor, Vorschlag vorhanden"}
for lage in ("Alle uneins", "Patt", "Mehrheit"):
    k = verteilung.get(lage, 0)
    if k or lage != "Patt":
        print(f"  {lage:<13} {k:>4}   ({erklaerung[lage]})")

alle_token = list(daten["token"])
for lage in ("Alle uneins", "Patt", "Mehrheit"):
    teil = unstimmig[unstimmig["lage"] == lage]
    if not len(teil):
        continue
    print(f"\n{'=' * 62}\n{lage} ({len(teil)})\n{'=' * 62}")
    for _, r in teil.head(12).iterrows():
        i = int(r["id"])
        links = " ".join(alle_token[max(0, i - 4):i])
        rechts = " ".join(alle_token[i + 1:i + 5])
        urteile = " | ".join(f"{p}: {r[p]}" for p in personen)
        marke = "  [unsicher gekennzeichnet]" if r["markiert"] else ""
        stimmen = f"{int(r['stimmen'])}/{len(personen)}"
        print(f"  '{r['token']}'  ({stimmen}, id {i}, {r['sammlung']}){marke}")
        print(f"      ... {links} [{r['token']}] {rechts} ...")
        print(f"      {urteile}"
              + (f"   Mehrheit: {r['mehrheit']}" if r["mehrheit"] else ""))
    if len(teil) > 12:
        print(f"  … {len(teil) - 12} weitere in der CSV")

export = unstimmig[["lage", "stimmen", "id", "token", "erzaehlung", "sammlung"]
                   + personen + ["mehrheit", "markiert"]].copy()
export["entscheidung"] = ""
export["begruendung"] = ""
ziel = AUSGABE / f"divergenzen_pos_t{TRANCHE}_lauf{LAUF_NR:02d}.csv"
export.to_csv(ziel, index=False, encoding="utf-8")
print(f"\nGeschrieben: {ziel}")
print("Die CSV ist nach Strittigkeit sortiert; Faelle ohne Mehrheit stehen oben.")


Divergenzen: 240 von 1794 Token

  Alle uneins     21   (jede Person ein anderes Tag, keine Mehrheit)
  Mehrheit       219   (eine Mehrheit liegt vor, Vorschlag vorhanden)

Alle uneins (21)
  'doch'  (1/3, id 118, Andersen)  [unsicher gekennzeichnet]
      ... auf einen Klumpen Erde [doch] der herausgefallen war und ...
      A4: ADV | A5: CCONJ | A6: PART
  'einer'  (1/3, id 186, Andersen)  [unsicher gekennzeichnet]
      ... eine zischelte dem andern [einer] , was das Kind ...
      A4: PRON | A5: NUM | A6: DET
  'aller'  (1/3, id 542, Andersen)
      ... wir verließen unter der [aller] des Königs und Bewahrers ...
      A4: DET | A5: ADJA | A6: PRON
  'doch'  (1/3, id 585, Andersen)
      ... ihn , ging auf [doch] Markt , wechselte ihn ...
      A4: ADV | A5: CCONJ | A6: PART
  'sicher'  (1/3, id 598, Andersen)  [unsicher gekennzeichnet]
      ... und gekochtes Fleisch ; [sicher] aß also mit den ...
      A4: ADV | A5: ADJD | A6: ADJA
  'wie'  (1/3, id 946, Arabisch)
      ... tröpf

## Schritt 9 — Kommentare der Bearbeitenden einsammeln


In [11]:
basis_index = set(vollstaendig.index)
strittige_index = set(unstimmig.index)
id_je_index = dict(zip(vollstaendig.index, vollstaendig["id"]))
token_je_index = dict(zip(vollstaendig.index, vollstaendig["token"]))

kommentare = []
for person in personen:
    spalte = tabellen[person].get("kommentar")
    if spalte is None:
        continue
    for idx, text in spalte.items():
        text = str(text).strip()
        if not text or idx not in basis_index:
            continue
        kommentare.append({
            "person": person,
            "id": int(id_je_index[idx]),
            "token": token_je_index[idx],
            "strittig": bool(idx in strittige_index),
            "kommentar": text,
        })

print(f"Kommentare insgesamt: {len(kommentare)}")
bei_strittig = [k for k in kommentare if k["strittig"]]
bei_einig    = [k for k in kommentare if not k["strittig"]]
print(f"  bei strittigen Einheiten:   {len(bei_strittig)}")
print(f"  bei einstimmigen Einheiten: {len(bei_einig)}  <- besonders pruefen")

if bei_einig:
    print()
    print("Kommentare trotz Einstimmigkeit (moegliche Regelluecke):")
    for k in bei_einig[:15]:
        print(f"  [{k['person']}] {k['token']}: {k['kommentar'][:70]}")
if bei_strittig:
    print()
    print("Kommentare bei strittigen Einheiten (Auszug):")
    for k in bei_strittig[:10]:
        print(f"  [{k['person']}] {k['token']}: {k['kommentar'][:70]}")

if kommentare:
    ziel_k = AUSGABE / f"kommentare_pos_t{TRANCHE}_lauf{LAUF_NR:02d}.csv"
    pd.DataFrame(kommentare).to_csv(ziel_k, index=False, encoding="utf-8")
    print(f"\nGeschrieben: {ziel_k}")
else:
    print("\nKeine Kommentare hinterlegt.")


Kommentare insgesamt: 176
  bei strittigen Einheiten:   75
  bei einstimmigen Einheiten: 101  <- besonders pruefen

Kommentare trotz Einstimmigkeit (moegliche Regelluecke):
  [A4] um: Oder doch ADP?
  [A4] schönste: P3: substantivierter Superlativ ohne folgendes Substantiv -> NOUN, abe
  [A4] Trauern: Substantiviertes Verb?
  [A4] Erstaunen: Substantiviertes Verb?
  [A4] ›Nimm: Zeichen ignorireren?
  [A4] essen.‹: Zeichen ignorieren?
  [A4] Dinaren: PROPN?
  [A4] begnügen.‹: (Satz-)Zeichen ignorieren?
  [A4] ›Der: (Satz-)Zeichen ignorieren?
  [A4] tot?‹: (Satz-)Zeichen ignorieren?
  [A4] ›Möge: (Satz-)Zeichen ignorieren?
  [A4] leben!‹: (Satz-)Zeichen ignorieren?
  [A4] ›Solches: (Satz-)Zeichen ignorieren?
  [A4] Habgierigen!‹: (Satz-)Zeichen ignorieren?
  [A4] Dinare: PROPN?

Kommentare bei strittigen Einheiten (Auszug):
  [A4] Nun: oder ADV?
  [A4] einer: P12: Partitiv 'einer der ...' ohne Bezugssubstantiv -> PRON; mechanisc
  [A4] wo: Relativadverb 'wo'; Anleitung hat keine eigene K

## Schritt 10 — Ergebnis sichern und einordnen


In [12]:
import json
from datetime import datetime

ergebnis = {
    "lauf": LAUF_NR, "etikett": LAUF_ETIKETT,
    "richtlinie_version": RICHTLINIE_VERSION, "tranche": TRANCHE,
    "zeitpunkt": datetime.now().isoformat(timespec="minutes"),
    "ebene": "pos", "bearbeitende": personen,
    "n_token_gesamt": int(len(daten)),
    "n_token_gemessen": int(len(vollstaendig)),
    "luecken": luecken,
    "hauptwert_zusammengefasst": grob,
    "fein_mit_adja_adjd": fein,
    "adjektiv_feindifferenzierung": adj_werte,
    "paarweise": paarwerte,
    "abweichungsindex": {k: float(v) for k, v in index.items()},
    "quote": float(p_quote),
    "cluster_bootstrap_95": [float(lo), float(hi)],
    "wilson_95": [float(zentrum - rand), float(zentrum + rand)],
    "nach_sammlung": {s: float(g["einheit_pa"].mean())
                      for s, g in vollstaendig.groupby("sammlung")},
    "divergenzen": int(len(unstimmig)),
}
ziel_json = AUSGABE / f"iaa_pos_t{TRANCHE}_lauf{LAUF_NR:02d}.json"
ziel_json.write_text(json.dumps(ergebnis, ensure_ascii=False, indent=2),
                     encoding="utf-8")
print(f"Geschrieben: {ziel_json}")

a_wert = grob["fleiss_kappa"]
print()
print(f"Fleiss' kappa (Hauptwert, zusammengefasst) = {a_wert:.3f} "
      f"(beobachtet {grob['beobachtet']:.3f}, "
      f"erwartet {grob['erwartet']:.3f})")
if a_wert >= 0.8:
    print("  Einordnung: gute Reliabilitaet (Konvention: ab 0,8).")
elif a_wert >= 0.67:
    print("  Einordnung: nur vorsichtige Schluesse zulaessig (0,67 bis 0,8).")
else:
    print("  Einordnung: unzureichend — Richtlinie und Einweisung pruefen.")
if adj_werte:
    print(f"Adjektiv-Feindifferenzierung: Fleiss kappa = "
          f"{adj_werte['fleiss_kappa']:.3f} "
          f"(n = {adj_werte['n_einig_dass_adjektiv']})")

print()
print("Alle bisherigen Laeufe dieser Tranche:")
for pfad in sorted(AUSGABE.glob(f"iaa_pos_t{TRANCHE}_lauf*.json")):
    d = json.loads(pfad.read_text(encoding="utf-8"))
    print(f"  Lauf {d['lauf']:>2} | {d['zeitpunkt']} | {d['etikett']:<22} | "
          f"Fleiss K {d['hauptwert_zusammengefasst']['fleiss_kappa']:.3f} | "
          f"Divergenzen {d['divergenzen']}")
if LAUF_NR > 1:
    print()
    print("HINWEIS: Massgeblich ist der Lauf VOR der Adjudikation (Lauf 1).")

Geschrieben: ../05_ergebnisse/iaa/iaa_pos_t1_lauf01.json

Fleiss' kappa (Hauptwert, zusammengefasst) = 0.896 (beobachtet 0.907, erwartet 0.103)
  Einordnung: gute Reliabilitaet (Konvention: ab 0,8).
Adjektiv-Feindifferenzierung: Fleiss kappa = 0.902 (n = 90)

Alle bisherigen Laeufe dieser Tranche:
  Lauf  1 | 2026-09-13T13:48 | vor Adjudikation       | Fleiss K 0.896 | Divergenzen 240
